In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('./dataset/synthetic_logs.csv')
df.head()

,timestamp,source,log_message,target_label
0,27-06-2025 07:20,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert
3,12-07-2025 00:24,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status
4,02-06-2025 18:25,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status


In [2]:
df.source.unique()

<ArrowStringArray>
[      'ModernCRM', 'AnalyticsEngine',        'ModernHR',   'BillingSystem',
   'ThirdPartyAPI',       'LegacyCRM']
Length: 6, dtype: str

In [3]:
df.target_label.unique()

<ArrowStringArray>
[        'HTTP Status',      'Critical Error',      'Security Alert',
               'Error', 'System Notification',      'Resource Usage',
         'User Action',      'Workflow Error', 'Deprecation Warning']
Length: 9, dtype: str

In [4]:
import os
from google import genai
from google.genai import types
from dotenv import load_dotenv
import time

# Load key from .env
load_dotenv('../backend/.env')
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

# Function to get embeddings in batches or one by one
def get_embeddings(texts, batch_size=100):
    embeddings = []
    start = time.time()
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        result = client.models.embed_content(
            model="models/gemini-embedding-001",
            contents=batch,
            config=types.EmbedContentConfig(task_type="CLASSIFICATION")
        )
        for emb in result.embeddings:
            embeddings.append(emb.values)
        if (i + batch_size) % 500 == 0 or i + batch_size >= len(texts):
            elapsed = time.time() - start
            done = min(i + batch_size, len(texts))
            rate = done / elapsed
            remaining = (len(texts) - done) / rate
            print(f"{done}/{len(texts)} done | ~{remaining/60:.1f} min remaining")
    print(f"Done! {len(texts)} embeddings in {(time.time()-start)/60:.1f} min")
    return embeddings

# Get new embeddings
# embeddings = get_embeddings(df['log_message'].tolist())
# embeddings

In [5]:
import re
def classify_with_regex(log_message):
    regex_patterns = {
        r"User User\d+ logged (in|out).": "User Action",
        r"Backup (started|ended) at .*": "System Notification",
        r"Backup completed successfully.": "System Notification",
        r"System updated to version .*": "System Notification",
        r"File .* uploaded successfully by user .*": "System Notification",
        r"Disk cleanup completed successfully.": "System Notification",
        r"System reboot initiated by user .*": "System Notification",
        r"Account with ID .* created by .*": "User Action"
    }
    for pattern, label in regex_patterns.items():
        if re.search(pattern, log_message,re.IGNORECASE):
            return label
    return None

In [6]:
classify_with_regex('User User123 logged OUT.')

'User Action'

In [7]:
df['regex_label'] = df['log_message'].apply(classify_with_regex)
df[df.regex_label.notnull()]

,timestamp,source,log_message,target_label,regex_label
7,10-11-2025 08:44,ModernHR,File data_6169.csv uploaded successfully by us...,System Notification,System Notification
14,01-04-2025 01:43,ThirdPartyAPI,File data_3847.csv uploaded successfully by us...,System Notification,System Notification
15,05-01-2025 09:41,ModernCRM,Backup completed successfully.,System Notification,System Notification
18,2/22/2025 17:49,ModernCRM,Account with ID 5351 created by User634.,User Action,User Action
27,9/24/2025 19:57,ThirdPartyAPI,User User685 logged out.,User Action,User Action
...,...,...,...,...,...
2376,6/27/2025 8:47,ModernCRM,System updated to version 2.0.5.,System Notification,System Notification
2381,09-05-2025 06:39,ThirdPartyAPI,Disk cleanup completed successfully.,System Notification,System Notification
2394,04-03-2025 13:13,ModernHR,Disk cleanup completed successfully.,System Notification,System Notification
2395,05-02-2025 14:29,ThirdPartyAPI,Backup ended at 2025-05-06 11:23:16.,System Notification,System Notification


In [8]:
df_non_regex = df[df['regex_label'].isnull()].copy()
df_non_regex

,timestamp,source,log_message,target_label,regex_label
0,27-06-2025 07:20,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,NaN
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,NaN
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,NaN
3,12-07-2025 00:24,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,NaN
4,02-06-2025 18:25,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,NaN
...,...,...,...,...,...
2405,13-08-2025 07:29,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,NaN
2406,01-11-2025 05:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,NaN
2407,03-08-2025 03:07,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,NaN
2408,11-11-2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,NaN


In [9]:
len(df_non_regex)

1910

In [10]:
print(df_non_regex['target_label'].value_counts()[df_non_regex['target_label'].value_counts() <= 5].index.tolist())

['Workflow Error', 'Deprecation Warning']


In [11]:
df_non_legacy = df_non_regex[df_non_regex.source!='LegacyCRM']
df_non_legacy.source.unique()

<ArrowStringArray>
['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem', 'ThirdPartyAPI']
Length: 5, dtype: str

In [12]:
len(df_non_legacy)

1903

In [13]:
# embeddings_filtered = model.encode(df_non_legacy['log_message'].tolist())
# embeddings_filtered[:2]

embeddings_filtered = get_embeddings(df_non_legacy['log_message'].tolist())
embeddings_filtered[:2]


500/1903 done | ~0.6 min remaining
1000/1903 done | ~0.4 min remaining
1500/1903 done | ~0.2 min remaining
1903/1903 done | ~0.0 min remaining
Done! 1903 embeddings in 0.8 min


[[-0.007538417,
  0.011511188,
  0.018621909,
  -0.06331058,
  0.0078004706,
  -0.0014331058,
  -0.003207318,
  0.02046956,
  0.0033328524,
  0.016715098,
  -0.014400018,
  -0.016864471,
  0.00091774133,
  0.020523915,
  0.120129265,
  -0.0029322156,
  -0.0033266777,
  0.01298224,
  0.008360264,
  -0.03611287,
  -0.0008582884,
  -0.0068909754,
  0.026716156,
  -0.010525538,
  -0.008323638,
  0.0024715094,
  0.023772037,
  0.008334851,
  0.05492315,
  -0.013746696,
  -0.02181729,
  0.008870961,
  0.019475857,
  0.009154856,
  0.009028968,
  0.0207672,
  0.02137762,
  -0.008664642,
  -0.0040736985,
  0.023102248,
  -0.0079638185,
  0.010258231,
  0.019246513,
  -0.004759065,
  0.012589283,
  0.022217069,
  0.010471957,
  -0.021365039,
  -0.006701783,
  0.026097754,
  -0.021573931,
  -0.007715729,
  -0.0040037585,
  -0.17945057,
  0.007701725,
  0.0118681565,
  0.0024491996,
  -0.009230214,
  0.028698882,
  0.0067891157,
  -0.010573268,
  0.02375107,
  0.0038178423,
  -0.01978984,
  -0.01

In [14]:
X = np.array(embeddings_filtered)
y = np.array(df_non_legacy['target_label'].values)

In [15]:
X.shape

(1903, 3072)

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,accuracy_score,confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [17]:
y_pred = clf.predict(X_test)
report = classification_report(y_test, y_pred)
print(report)

                precision    recall  f1-score   support

Critical Error       0.98      1.00      0.99        48
         Error       1.00      0.98      0.99        47
   HTTP Status       1.00      1.00      1.00       304
Resource Usage       1.00      1.00      1.00        49
Security Alert       1.00      1.00      1.00       123

      accuracy                           1.00       571
     macro avg       1.00      1.00      1.00       571
  weighted avg       1.00      1.00      1.00       571



In [18]:
print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy Score: 0.9982486865148862
Confusion Matrix:
 [[ 48   0   0   0   0]
 [  1  46   0   0   0]
 [  0   0 304   0   0]
 [  0   0   0  49   0]
 [  0   0   0   0 123]]


In [19]:
import joblib
joblib.dump(clf, '../backend/models/log_classifier.joblib')

['../backend/models/log_classifier.joblib']